In [4]:
Sys.setenv(
  http_proxy  = "http://100.83.45.1:7897",
  https_proxy = "http://100.83.45.1:7897",
  all_proxy   = "socks5://100.83.45.1:7897",
  no_proxy    = "localhost,127.0.0.1"
)
options(timeout = 3600)

↑ 如果直接尝试下载，会因为网络原因失败，疑似需要代理

本机暂未部署代理，但加入了另外几台机器的 tailscale 网络，而其中一台机器有代理（100.83.45.1:7897），本次使用该机器的代理下载数据（在notebook顶端设置）

In [5]:
library(spatialLIBD)
library(ExperimentHub)

In [6]:
ehub <- ExperimentHub()
spe <- fetch_data(type = "spe", eh = ehub)

adding rname 'https://www.dropbox.com/s/f4wcvtdq428y73p/Human_DLPFC_Visium_processedData_sce_scran_spatialLIBD.Rdata?dl=1'



2026-10-14 13:47:49.969326 loading file /home/liuzy/.cache/R/BiocFileCache/1712f3e97c3c3_Human_DLPFC_Visium_processedData_sce_scran_spatialLIBD.Rdata%3Fdl%3D1



In [13]:
table(spe$sample_id)


151507 151508 151509 151510 151669 151670 151671 151672 151673 151674 151675 
  4226   4384   4789   4634   3661   3498   4110   4015   3639   3673   3592 
151676 
  3460 

<span style="color: red">新数据（spe）与旧 h5（sample_151507_anno.h5）存在差异</span>。以坐标为例：旧文件的 Pos 是 Visium 的 array 网格坐标 (array_row, array_col)；而 spe 默认提供的坐标是 spatialCoords(spe)（全分辨率像素坐标）。好在 spe 并未丢失 array 坐标，它保留在 colData 的 array_row/array_col 中，只是需要自行取出。要复刻旧文件的 Pos，应取 spe$array_row/array_col；再考虑到 h5write 与 h5py 的维序相反（旧文件在 h5py 下 Pos 为 (2,n)），R 端需写成 cbind(spe$array_row[sel], spe$array_col[sel])（n×2），不要 t(cbind(...))。

In [43]:
# 旧文件里的 Pos/Loc 是什么样的？
library(rhdf5)
f_old <- "../../data/sample_151507_anno.h5"
old_Pos <- h5read(f_old, "Pos")
old_Loc <- h5read(f_old, "Loc")
dim(old_Pos); apply(old_Pos, 2, range) # 4226 x 2 ; 第1列 0..77，第2列 0..127（整数网格）
dim(old_Loc); apply(old_Loc, 2, range) # 4226 x 2 ; 2k..11k（带小数，像素坐标）
# head(old_Pos); head(old_Loc)

[1] 4226    2

0,0
77,127


[1] 4226    2

2513,2160
11751,10917


In [45]:
# spe 的 151507 子集里的默认坐标：全分辨率像素坐标
sel <- spe$sample_id == "151507"
pixel_pos <- spatialCoords(spe)[sel, ] # 全分辨率像素坐标
apply(pixel_pos, 2, range) # 2k..11k，同 old_Loc，像素坐标

pxl_col_in_fullres,pxl_row_in_fullres
2160,2513
10917,11751


[1] "Mean relative difference: 0.4139723"

要复刻旧文件的 Pos，用 spe$array_row/array_col，新数据的 spatialCoords() 不能作为 DSSC 的输入。

In [47]:
array_pos <- cbind(spe$array_row[sel], spe$array_col[sel])
apply(array_pos, 2, range) # 0..77 / 0..127，同 old_Pos

0,0
77,127


In [49]:
all.equal(unname(old_Pos), unname(array_pos)) # TRUE，旧 Pos 就是 array_row/array_col
all.equal(unname(old_Pos), unname(pixel_pos)) # 旧 Pos 不是像素坐标

[1] TRUE

[1] "Mean relative difference: 132.5765"

旧 Pos 用的是 Visium 网格坐标（array_row/array_col），不用像素坐标；又因为 h5write/h5py 的维序相反，要在 h5py 里得到旧文件那样的 (2, n)，R 端就必须写成行数=spot 的 cbind(array_row, array_col)，不是 t(cbind(...))，见下文。

In [18]:
# 将 spe 按 sample_id 拆成 12 个 DSSC 输入格式的 h5
# X: spots x genes 计数; Y: 真值标签(Layer1..6/WM/NA); Pos: spots x 2; Gene: 基因名
outdir <- "../../data/spatialLIBD"
dir.create(outdir, showWarnings = FALSE, recursive = TRUE)
cnt <- SummarizedExperiment::assay(spe, "counts")
genes <- SummarizedExperiment::rowData(spe)$gene_name
coord <- SpatialExperiment::spatialCoords(spe)
lab <- as.character(spe$layer_guess_reordered)
ids <- spe$sample_id
for (s in unique(ids)) {
    i <- which(ids == s)
    X <- as.matrix(cnt[, i, drop = FALSE]) 
    Y <- lab[i]; Y[is.na(Y)] <- "NA"
    Pos <- cbind(spe$array_row[i], spe$array_col[i]) #! R: spots x 2 -> h5py: (2, spots)
    fp <- file.path(outdir, paste0("sample_", s, "_anno.h5"))
    if (file.exists(fp)) file.remove(fp)
    rhdf5::h5createFile(fp)
    rhdf5::h5write(X, fp, "X")
    rhdf5::h5write(Y, fp, "Y")
    rhdf5::h5write(Pos, fp, "Pos")
    rhdf5::h5write(genes, fp, "Gene")
    message("wrote ", fp, "  X=", paste(dim(X), collapse = "x"))
}

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.1 GiB”
You created a large dataset with compression and chunking.
The chunk size is equal to the dataset dimensions.
If you want to read subsets of the dataset, you should test smaller chunk sizes to improve read times.

wrote ../../data/spatialLIBD/sample_151507_anno.h5  X=33538x4226

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.1 GiB”
You created a large dataset with compression and chunking.
The chunk size is equal to the dataset dimensions.
If you want to read subsets of the dataset, you should test smaller chunk sizes to improve read times.

wrote ../../data/spatialLIBD/sample_151508_anno.h5  X=33538x4384

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.2 GiB”
You created a large dataset with compression and chunking.
The chunk size is equal to the dataset dimensions.
If you want to read subsets of the data